<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Resposta:** Não é obrigatório realizar a normalização para algoritmos baseados em árvores, como Random Forest e AdaBoost. Isso ocorre porque esses modelos tomam decisões de partição baseadas em limiares (thresholds) individuais de cada feature, o que os torna indiferentes à escala global dos dados. Diferente de modelos que medem distâncias euclidianas (como KNN), o fato de um pixel variar de 0 a 255 não distorce a lógica de quebra do nó. Contudo, normalizar os dados dividindo por 255 é frequentemente recomendado para manter o pipeline compatível com outros estimadores sensíveis à escala, como Redes Neurais.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

def load_data(seed=42):
    X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')
    y = y.astype(int)
    
    return train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

X_train, X_test, y_train, y_test = load_data(42)


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(random_state=seed, n_jobs=-1)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc


Para validar o desempenho, a função `evaluate` calcula a taxa de acerto global. Como o Fashion MNIST possui classes balanceadas, a acurácia é um indicador primário confiável para verificar o quão bem o modelo generaliza para dados não vistos durante o treinamento.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    acc = evaluate(model, X_test, y_test)
    return acc

acc_rf_demo = run_pipeline("rf", 42)
print(f"Acurácia RF Demo (Seed 42): {acc_rf_demo:.4f}")


**Em qual profundidade começa o overfitting?**
O sobreajuste costuma surgir quando as ramificações da árvore capturam detalhes excessivamente específicos (ruídos) do treino. No Random Forest, embora o bagging ajude a estabilizar o erro, árvores excessivamente profundas começam a perder poder de generalização ao tentar classificar perfeitamente cada exemplo atípico.

**Por que a árvore consegue 100% no treino quando max_depth=None?**
Sem um limite de profundidade, o algoritmo continua dividindo os nós até que todas as amostras em uma folha pertençam à mesma classe. Isso resulta em uma memorização literal dos dados de treinamento, atingindo a precisão total naquele conjunto específico.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Resposta**: O Random Forest demonstrou uma performance inicial superior. Isso se justifica pela sua estrutura de árvores independentes e profundas, que conseguem modelar melhor a complexidade visual das imagens desde o início. O AdaBoost, operando sequencialmente com estimadores fracos (geralmente árvores muito rasas por padrão), exige mais iterações ou ajustes para alcançar a mesma capacidade de representação que o ensemble de bagging oferece.

**Solução**:

In [ ]:
from sklearn.metrics import classification_report

print("Executando pipeline para RF e AdaBoost...")
X_train, X_test, y_train, y_test = load_data(42)

rf_model = train_random_forest(X_train, y_train, 42)
rf_acc = evaluate(rf_model, X_test, y_test)

ab_model = train_adaboost(X_train, y_train, 42)
ab_acc = evaluate(ab_model, X_test, y_test)

print(f"Random Forest - Acurácia: {rf_acc:.4f}")
print(f"AdaBoost - Acurácia: {ab_acc:.4f}")

print("\nClassification Report RF:")
print(classification_report(y_test, rf_model.predict(X_test)))

print("\nClassification Report AB:")
print(classification_report(y_test, ab_model.predict(X_test)))

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.


**Resposta:** Sim, o procedimento é reprodutível. A consistência dos resultados é garantida pelo uso do parâmetro `random_state`. Ao fixar uma semente, as flutuações estocásticas do treinamento e da divisão de dados tornam-se determinísticas, permitindo que qualquer pesquisador obtenha os mesmos valores exatos ao rodar o código com a mesma configuração.

**Solução**:

In [ ]:
acc_42 = run_pipeline("rf", 42)
acc_7 = run_pipeline("rf", 7)
print(f"Acurácia (Seed 42): {acc_42:.4f}")
print(f"Acurácia (Seed 7): {acc_7:.4f}")


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting? 


**Resposta:** Sim, há indícios claros de overfitting, visto que o desempenho no conjunto de treino foi superior ao do teste.
- Qual modelo tende a sofrer mais com isso? 


**Resposta:** O AdaBoost costuma ser mais vulnerável ao overfitting caso o número de estimadores seja muito elevado ou os dados possuam ruído excessivo, pois ele foca agressivamente na correção dos erros das árvores anteriores. Já o Random Forest é naturalmente mais resiliente ao sobreajuste, pois a técnica de Bagging reduz a variância global ao calcular a média de múltiplas árvores construídas de forma independente.

In [ ]:
rf_train_acc = rf_model.score(X_train, y_train)
rf_test_acc = rf_model.score(X_test, y_test)
print(f"RF Treino: {rf_train_acc:.4f}, Teste: {rf_test_acc:.4f}")

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

**Resposta**: O AdaBoost demonstra maior sensibilidade a alterações de hiperparâmetros. Como o aprendizado é sequencial (uma etapa depende da anterior), pequenas mudanças no número de estimadores ou na complexidade da base podem causar um impacto em cascata no resultado final. No Random Forest, como o processo é paralelo e baseado em média, a adição ou remoção de algumas árvores tende a produzir variações mais suaves no desempenho.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

for n in [50, 100]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    print(f"RF (n_estimators={n}): {rf.score(X_test, y_test):.4f}")

for n in [50, 100]:
    ab = AdaBoostClassifier(n_estimators=n, random_state=42)
    ab.fit(X_train, y_train)
    print(f"AB (n_estimators={n}): {ab.score(X_test, y_test):.4f}")


# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

### Respostas da Questão 9:

1. **A acurácia é suficiente?**
Não totalmente. Embora útil em datasets balanceados, a acurácia é uma métrica simplista que não distingue o tipo de erro cometido. Ela não revela, por exemplo, se o modelo está tendo dificuldades específicas com uma categoria (como diferenciar 'casacos' de 'camisas'), o que exigiria a análise de Precisão e Recall.

2. **Como garantir que o resultado não ocorreu por acaso?**
A garantia advém do uso rigoroso de sementes aleatórias fixas (`random_state`). Isso elimina a incerteza estatística entre execuções, assegurando que o resultado observado é fruto da lógica do modelo e da divisão de dados definida, e não de uma inicialização fortuita.

3. **Problemas metodológicos:**
- **Ausência de Otimização:** O experimento utiliza parâmetros padrão, o que impede de conhecer o teto real de performance de cada algoritmo através de um Grid Search.
- **Divisão Estática:** O uso de apenas um split (treino/teste) pode gerar resultados enviesados por aquela partição específica. O ideal seria aplicar Validação Cruzada (K-Fold) para obter uma média de desempenho mais robusta.

4. **O pipeline é confiável?**
Sim. Ele segue boas práticas de Ciência de Dados ao isolar completamente os dados de teste durante a fase de treino, evitando o vazamento de informação (data leakage). Sua estrutura modular e o controle de aleatoriedade o tornam um processo transparente e facilmente auditável.